### Corte en longitudes de onda para definir intervalo común

El objetivo de este código es cortar los espectros entre los valores min y max de loglam(encontrados con el algoritmo (max-min_spec)) y seleccionar los primeros 3600 valores, con el fin de unificar la ventana de longitudes de onda.

In [ ]:
import os
from astropy.io import fits

In [ ]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


Valores encontrados:

Máximo valor de valores mínimos: 3.5844

Mínimo valor de valores máximos: 3.963




In [ ]:
#path a la carpeta donde estan los espectros y donde se quiere guardarlos después de cortarlos
dir = '/drive/My Drive/Data/raw_spec/'

output_dir = '/drive/My Drive/Data/spec_data/cut_spec'
os.makedirs(output_dir, exist_ok=True)



In [ ]:
#!ls '/drive/My Drive/Data/raw_spec'

In [ ]:
spec_files = len(os.listdir(dir))
print(f'Hay {spec_files} archivos en la carpeta.')

Hay 128 archivos en la carpeta.


In [ ]:
#especificar los parámetros para filtrar los espectros
col_name = 'loglam'
min_val = 3.5844
max_val = 3.9630

In [ ]:
#crear la lista de espectros (.fits) en la carpeta
fits_files = [os.path.join(dir, file) for file in os.listdir(dir)
    if file.startswith('spec') and file.endswith('.fits')]


In [ ]:
#procesar los espectros
for fits_file in fits_files:
    #print(f"Procesando archivo: {fits_file}")
    output_file = os.path.join(output_dir, f'cortado_{os.path.basename(fits_file)}')

    #abrir los espectros
    hdul = fits.open(fits_file)

    if len(hdul) > 1 and hasattr(hdul[1], 'data'): #verificar si el archivo tiene mas de una extensión y que la segunda extensión es la que tiene datos
        data = hdul[1].data #asignarle los valores de la extensión a la variable data
        if col_name in data.columns.names:

            #crear una mascara para filtrar valores de las long de onda (loglam) entre el valor mínimo y máximo
            mask = (data[col_name] >= min_val) & (data[col_name] <= max_val)

            #aplicar la mascara a la tabla, para conservar solo los valores entre (valor min, valor max)
            filtered_data = data[mask]

             #conservar solo los primeros 3600 valores
            if len(filtered_data) > 3600:
                filtered_data = filtered_data[:3600]

            #verificar si hay datos desp de aplicar el filtro
            if len(filtered_data) > 0:

                #guardar el espectr filtrado si hay datos dentro del rango
                hdu = fits.BinTableHDU(filtered_data)
                os.makedirs(os.path.dirname(output_file), exist_ok=True)  #crear la carpeta donde se guardan los espectros filtrados
                hdu.writeto(output_file, overwrite=True)
                #print(f"Espectro filtrado guardado en {output_file}")
            else:
                #en el caso de que no haya valores entre ese rango de valores
                print(f'El espectro {fits_file} no contiene valores entre {min_val} y {max_val}.')

    hdul.close()



In [ ]:
total_files = len(os.listdir(output_dir))
print(f'Hay {total_files} archivos en la carpeta {output_dir}.')

Hay 129 archivos en la carpeta.
